---
**Script name:** 04_bat_data_classification_filtering.ipynb

**Purpose of script:** This script is to filter data according to it's activity classification. 

**Author:** Dr Katie Allan

**Date Created:** 2023-02-19

**Email:** katie.allan@sussex.ac.uk

**License:** 

---

**Notes:** Ensure folder paths are set properly.

---

## Importing libraries that are required

In [ ]:
import time
from datetime import timedelta
import sqlalchemy as psql
import pickle
import pandas as pd
import numpy as np
import os

## Functions that are used throughout the script

In [ ]:
def bat_sort(dataframe, bat_column):
    #create empty python dictionary
    bat_dic = {}
    
    #create list of unique bat tags 
    unique_bats = sorted(dataframe[bat_column].unique())
    
    for bat in range(len(unique_bats)):
        #set bat name
        bat_name = unique_bats[bat]
        
        #filter dataframe by bat name
        bat_dataframe = dataframe[dataframe[bat_column] == bat_name].sort_values(by = 'time')
        
        #assign to dictionary
        bat_dic[bat_name] = bat_dataframe
        
    return bat_dic

In [ ]:
def apply_rolling_window(dataframes_dict):
    for key, df in dataframes_dict.items():
        # Initialize the 'window_period' column
        df['window_period'] = None
        
        # Iterate over the DataFrame and find windows
        for i in range(len(df)):
            if df.loc[i, 'activity_class'] == 'passive':
                # Find the start of the window
                start_idx = i
                
                # Find the last 'passive' before the first 'active' row
                end_idx = None
                for j in range(start_idx + 1, len(df)):
                    if df.loc[j, 'activity_class'] == 'active':
                        # The end of the window is the last 'passive' before 'active'
                        end_idx = j - 1
                        break

                # If no 'active' is found, use the last 'passive' row in the range
                if end_idx is None:
                    # Filter for 'passive' rows in the original DataFrame, starting from start_idx
                    passive_rows = df.loc[start_idx:][df.loc[start_idx:]['activity_class'] == 'passive']
                    
                    # Get the last 'passive' row's index from the original DataFrame
                    if not passive_rows.empty:
                        end_idx = passive_rows.index[-1]
                
                # Calculate the time difference for this window
                time_diff = df.loc[end_idx, 'time'] - df.loc[start_idx, 'time']
                
                # If the time difference is greater than 1 minute 30seconds, mark 'remove'
                if time_diff > pd.Timedelta(seconds=90):
                    df.loc[start_idx:end_idx, 'window_period'] = 'remove'

        # Store the modified dataframe back into the dictionary
        dataframes_dict[key] = df

    return dataframes_dict

## Load Data

In [ ]:
#connect to postgres database
bat_data = pd.read_csv('output/bat_data_filtered_classification_joined.csv')

In [ ]:
#show the top rows of the dataframe
bat_data

,radio_id,tag_id,node_id,tag_rssi,validated,time,station_id,method,data_size,median_error_seconds,...,bat_name,ring_no,capture_site,Date,daytime,week,max_rssi,n_receivers,activity_class,rec_type
0,5,19334B78,3750b3,-85.0,0,2023-02-16 12:49:19+00:00,None,manual,170094.0,NaN,...,Abby,B1982,rock_farm,2023-02-16,12:49:19,7,NaN,NaN,none,Node
1,5,19334B78,3750b3,-88.0,0,2023-02-16 12:49:21+00:00,None,manual,170094.0,NaN,...,Abby,B1982,rock_farm,2023-02-16,12:49:21,7,NaN,NaN,none,Node
2,5,19334B78,3750b3,-93.0,0,2023-02-16 12:49:23+00:00,None,manual,170094.0,NaN,...,Abby,B1982,rock_farm,2023-02-16,12:49:23,7,NaN,NaN,none,Node
3,5,19334B78,3750b3,-73.0,0,2023-02-16 12:49:25+00:00,None,manual,170094.0,NaN,...,Abby,B1982,rock_farm,2023-02-16,12:49:25,7,NaN,NaN,none,Node
4,5,19334B78,3750b3,-80.0,0,2023-02-16 12:49:36+00:00,None,manual,170094.0,NaN,...,Abby,B1982,rock_farm,2023-02-16,12:49:36,7,NaN,NaN,none,Node
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1191249,5,4C2A4B4B,3750b3,-67.0,0,2023-02-20 23:55:37+00:00,None,manual,170094.0,NaN,...,Zeus,B1981,rock_farm,2023-02-20,23:55:37,8,NaN,NaN,none,Node
1191250,5,4C2A4B4B,3750b3,-66.0,0,2023-02-20 23:55:47+00:00,None,manual,170094.0,NaN,...,Zeus,B1981,rock_farm,2023-02-20,23:55:47,8,NaN,NaN,none,Node
1191251,5,4C2A4B4B,3750b3,-67.0,0,2023-02-20 23:55:58+00:00,None,manual,170094.0,NaN,...,Zeus,B1981,rock_farm,2023-02-20,23:55:58,8,NaN,NaN,none,Node
1191252,5,4C2A4B4B,3750b3,-66.0,0,2023-02-20 23:56:08+00:00,None,manual,170094.0,NaN,...,Zeus,B1981,rock_farm,2023-02-20,23:56:08,8,NaN,NaN,none,Node


In [ ]:
#check column types of each variable
bat_data.dtypes

radio_id                             object
tag_id                               object
node_id                              object
tag_rssi                            float64
validated                             int64
time                    datetime64[ns, UTC]
station_id                           object
method                               object
data_size                           float64
median_error_seconds                float64
max_error_seconds                   float64
latitude                            float64
longitude                           float64
deploy                  datetime64[ns, UTC]
remove                  datetime64[ns, UTC]
location                             object
software                             object
bat_name                             object
ring_no                              object
capture_site                         object
Date                         datetime64[ns]
daytime                              object
week                            

## Data processing

In [ ]:
#Change timezone of deploy and remove
bat_data['deploy'] = bat_data['deploy'].dt.tz_convert('Europe/London')
bat_data['remove'] = bat_data['deploy'].dt.tz_convert('Europe/London')

In [ ]:
#create new date and daytime columns
bat_data['Date'] =  pd.to_datetime(bat_data['time'].dt.date, format='%Y-%m-%d', errors='coerce')
bat_data['daytime'] = bat_data['time'].dt.time

In [ ]:
# create categorical column of activity_class
bat_data['activity_class'] = pd.Categorical(bat_data['activity_class'], ["active", "passive", "none"])

In [ ]:
#sort bats into a dictionary
bat_dic = bat_sort(bat_data, 'bat_name')

In [ ]:
# remove empty dataframes from dictionary
bat_dic = {key:df for (key,df) in bat_dic.items() if not df.empty}

## Apply rolling window to classify unclassified points

In [ ]:
# filter to only keep one row per timestamp, prioritise keeping active rows
bat_dic_reduced = {key: df.sort_values(['time','activity_class']).groupby('time').head(1).reset_index(drop=True) for key, df in bat_dic.items()}

In [ ]:
# Apply rolling window logic
bat_dic_reduced = apply_rolling_window(bat_dic_reduced)

# Check the modified dataframes
for key, df in bat_dic_reduced.items():
    print(f"DataFrame {key}:\n", df)

DataFrame Abby:
      radio_id    tag_id node_id  tag_rssi  validated  \
0           5  19334B78  3750b3     -85.0          0   
1           5  19334B78  3750b3     -88.0          0   
2           5  19334B78  3750b3     -93.0          0   
3           5  19334B78  3750b3     -73.0          0   
4           5  19334B78  3750b3     -80.0          0   
...       ...       ...     ...       ...        ...   
3312        5  19334B78  3750b3    -104.0          0   
3313        5  19334B78  3750b3    -105.0          0   
3314        5  19334B78  3750b3    -104.0          0   
3315        5  19334B78  3750b3    -104.0          0   
3316        5  19334B78  3750b3    -103.0          0   

                          time station_id  method  data_size  \
0    2023-02-16 12:49:19+00:00       None  manual   170094.0   
1    2023-02-16 12:49:21+00:00       None  manual   170094.0   
2    2023-02-16 12:49:23+00:00       None  manual   170094.0   
3    2023-02-16 12:49:25+00:00       None  manual   17

In [ ]:
bat_dic_reduced.keys()

dict_keys(['Abby', 'Andrea', 'Bert', 'Blob', 'Charlie', 'Conchita', 'Derek', 'Erik', 'Fred', 'George', 'Gerty', 'Heather', 'Ian', 'Iris', 'Isla', 'Jennet', 'Jez', 'Joyce', 'Kay', 'Larry', 'Laticia', 'Mary', 'May', 'Moira', 'Nancy', 'Neil', 'Norman', 'Olga', 'Olivia', 'Pedro', 'Polly', 'Queenie', 'Quincey', 'Rex', 'Ross', 'Steve', 'Strachy', 'Tony', 'Tracy', 'Uri', 'Usher', 'Vera', 'Vincent', 'Wally', 'Wilma', 'Xanthi', 'Xray', 'Yasmin', 'Yvette', 'Zac', 'Zeus'])

In [ ]:
#turn back into one big dataframe
all_bat_classification_reduced = pd.concat(bat_dic_reduced.values(), ignore_index=True)
all_bat_classification_reduced

,radio_id,tag_id,node_id,tag_rssi,validated,time,station_id,method,data_size,median_error_seconds,...,ring_no,capture_site,Date,daytime,week,max_rssi,n_receivers,activity_class,rec_type,window_period
0,5,19334B78,3750b3,-85.0,0,2023-02-16 12:49:19+00:00,None,manual,170094.0,NaN,...,B1982,rock_farm,2023-02-16,12:49:19,7,NaN,NaN,none,Node,None
1,5,19334B78,3750b3,-88.0,0,2023-02-16 12:49:21+00:00,None,manual,170094.0,NaN,...,B1982,rock_farm,2023-02-16,12:49:21,7,NaN,NaN,none,Node,None
2,5,19334B78,3750b3,-93.0,0,2023-02-16 12:49:23+00:00,None,manual,170094.0,NaN,...,B1982,rock_farm,2023-02-16,12:49:23,7,NaN,NaN,none,Node,None
3,5,19334B78,3750b3,-73.0,0,2023-02-16 12:49:25+00:00,None,manual,170094.0,NaN,...,B1982,rock_farm,2023-02-16,12:49:25,7,NaN,NaN,none,Node,None
4,5,19334B78,3750b3,-80.0,0,2023-02-16 12:49:36+00:00,None,manual,170094.0,NaN,...,B1982,rock_farm,2023-02-16,12:49:36,7,NaN,NaN,none,Node,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1191249,5,4C2A4B4B,3750b3,-67.0,0,2023-02-20 23:55:37+00:00,None,manual,170094.0,NaN,...,B1981,rock_farm,2023-02-20,23:55:37,8,NaN,NaN,none,Node,None
1191250,5,4C2A4B4B,3750b3,-66.0,0,2023-02-20 23:55:47+00:00,None,manual,170094.0,NaN,...,B1981,rock_farm,2023-02-20,23:55:47,8,NaN,NaN,none,Node,None
1191251,5,4C2A4B4B,3750b3,-67.0,0,2023-02-20 23:55:58+00:00,None,manual,170094.0,NaN,...,B1981,rock_farm,2023-02-20,23:55:58,8,NaN,NaN,none,Node,None
1191252,5,4C2A4B4B,3750b3,-66.0,0,2023-02-20 23:56:08+00:00,None,manual,170094.0,NaN,...,B1981,rock_farm,2023-02-20,23:56:08,8,NaN,NaN,none,Node,None


In [ ]:
len(all_bat_classification_reduced[all_bat_classification_reduced['window_period'] == 'remove'])

737716

In [ ]:
len(all_bat_classification_reduced) - len(all_bat_classification_reduced[all_bat_classification_reduced['window_period'] == 'remove'])

453538

## Export data

In [ ]:
#export data to csv file
all_bat_classification_reduced.to_csv('output/bat_data_filtered_classification_joined_reduced.csv', index=False)